In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# definição de configuração de taxa de conversão da moeda
TAXA_CONVERSAO = 0.00031 # rupia indonesia para brl

# Dicionários de Tradução (Mapas)
# Status do Pedido (Status_Pesanan)
status_map = F.create_map([
    F.lit("Selesai"), F.lit("Concluído"),
    F.lit("Dibatalkan"), F.lit("Cancelado"),
    F.lit("Sedang Dikirim"), F.lit("Em Entrega"),
    F.lit("Refund"), F.lit("Reembolsado")
])

# Nível de Reclamação (Tingkat_Keluhan)
complaint_map = F.create_map([
    F.lit("Tidak Ada"), F.lit("Nenhuma"),
    F.lit("Rendah"), F.lit("Baixa"),
    F.lit("Sedang"), F.lit("Média"),
    F.lit("Tinggi"), F.lit("Alta")
])

# menu
menu_map = F.create_map([
    F.lit("Kopi"), F.lit("Café"),
    F.lit("Mie"), F.lit("Macarrão"),
    F.lit("Martabak"), F.lit("Martabak (Panqueca Indonésia)"),
    F.lit("Ayam"), F.lit("Frango")
])

# Carrega os dados da bronze
df_bronze = spark.table('foodDelivery.staging.orders_synthetic')

# Calcula a media para inputação (Jarak_Kirim_KM)
median_distance = df_bronze.approxQuantile("Jarak_Kirim_KM", [0.5], 0.01)[0]

# Transformação inicial
df_silver = df_bronze.select(
        F.col("ID_Pesanan").alias("order_id"),

        # Padronização de data
        F.when(F.col("Waktu_Transaksi").contains("/"), 
             F.to_timestamp(F.trim(F.col("Waktu_Transaksi")), "dd/MM/yyyy HH:mm")
             ).otherwise(F.col("Waktu_Transaksi")
        ).alias("transaction_time"), 

        # tradução da categoria
        F.coalesce(menu_map[F.col("Kategori_Menu")], F.col("Kategori_Menu")).alias("menu_category"), 

        # valores financeiros
        F.round(F.col("Harga_Pesanan").cast(DecimalType(10, 2)), 2).alias("order_price"),
        F.round(F.col("Harga_Pesanan").cast(DecimalType(10, 2)) * F.lit(TAXA_CONVERSAO), 2).alias("order_price_brl"),

        # input de mediana (Jarak_Kirim_KM)
        F.coalesce(F.col("Jarak_Kirim_KM"), F.lit(median_distance)).alias("shipping_distance_km"),
        F.col("Waktu_Tunggu_Menit").alias("waiting_time_minutes"),
        F.col("Rating_Pelanggan").alias("customer_rating"),
        F.col("Ulasan_Teks").alias("review_text"),
        F.col("Status_Promo").alias("promo_status"),
        
        # Aplicação das Traduções
        F.coalesce(complaint_map[F.col("Tingkat_Keluhan")], F.col("Tingkat_Keluhan")).alias("complaint_level"),
        F.coalesce(status_map[F.col("Status_Pesanan")], F.col("Status_Pesanan")).alias("order_status")
) 

# cálculo dinâmico dos outliers

# calcula os limites estatisticos de cada categoria
df_stats = df_silver.groupBy("menu_category").agg(
    F.percentile_approx("order_price_brl", 0.25).alias("q1"),
    F.percentile_approx("order_price_brl", 0.75).alias("q3")
).withColumn("iqr", F.col("q3") - F.col("q1")) \
 .withColumn("lower_bound", F.when(F.col("q1") - 1.5 * F.col("iqr") < 0, 0).otherwise(F.col("q1") - 1.5 * F.col("iqr"))) \
 .withColumn("upper_bound", F.col("q3") + 1.5 * F.col("iqr"))

 # Join e aplicação do Flag
df_silver = df_silver.join(
    df_stats.select("menu_category", "lower_bound", "upper_bound"), 
    on="menu_category", 
    how="left"
)

df_silver = df_silver.withColumn(
    "is_outlier",
    F.when(
        (F.col("order_price_brl") < F.col("lower_bound")) | 
        (F.col("order_price_brl") > F.col("upper_bound")), 
        True
    ).otherwise(False)
).drop("lower_bound", "upper_bound")

# Salvar na camada Silver
(df_silver.write
 .format('delta')
 .mode('overwrite')
 .option("mergeSchema", "true")
 .saveAsTable('foodDelivery.silver.orders'))

dbutils.notebook.exit("SUCCESS")